# Georeference Image

Take a single image (.JPG) taken from a drone and attempt to calculate georeferences.

1. Extract GPS metadata
2. Transform the Image
3. Georeference the image

In [1]:
# file_path = '../input/DJI_0010.JPG'
# file_path = '../input/DJI_0015.JPG'
file_path = '../input/DJI_0093.JPG'
# file_path = '../input/DJI_0119.JPG'

In [6]:
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

def get_exif_data(image_path):
    image = Image.open(image_path)
    exif_data = image._getexif()
    gps_data = {}
    if exif_data:
        for tag, value in exif_data.items():
            decoded = TAGS.get(tag, tag)
            if decoded == "GPSInfo":
                for gps_tag, gps_value in value.items():
                    gps_decoded = GPSTAGS.get(gps_tag, gps_tag)
                    gps_data[gps_decoded] = gps_value
                # Extract GPS Altitude if available
                if "GPSAltitude" in gps_data:
                    altitude = float(gps_data["GPSAltitude"].numerator) / float(gps_data["GPSAltitude"].denominator)

    return gps_data, altitude

def convert_to_degrees(value):
    """Converts GPS coordinates stored as (degrees, minutes, seconds) into decimal degrees."""
    def rational_to_float(rational):
        return float(rational.numerator) / float(rational.denominator)

    d = rational_to_float(value[0])  # Degrees
    m = rational_to_float(value[1])  # Minutes
    s = rational_to_float(value[2])  # Seconds
    return d + (m / 60.0) + (s / 3600.0)

gps_data, altitude = get_exif_data(file_path)

latitude = convert_to_degrees(gps_data["GPSLatitude"])
longitude = convert_to_degrees(gps_data["GPSLongitude"])

if gps_data["GPSLatitudeRef"] == "S":
    latitude = -latitude
if gps_data["GPSLongitudeRef"] == "W":
    longitude = -longitude

print(f"Latitude: {latitude}, Longitude: {longitude}")

if altitude:
    print(f"Altitude: {altitude} meters")
else:
    print("Altitude information not found in the GPS metadata.")

Latitude: 30.249954083333336, Longitude: -103.60311508333332
Altitude: 1508.166 meters


In [5]:
import math

def calculate_ground_coverage_and_resolution(altitude, fov_horizontal, fov_vertical, image_width, image_height):
    """
    Calculate the ground coverage and pixel resolution of an image.

    Parameters:
        altitude (float): Altitude of the drone in meters.
        fov_horizontal (float): Horizontal field of view of the camera in degrees.
        fov_vertical (float): Vertical field of view of the camera in degrees.
        image_width (int): Width of the image in pixels.
        image_height (int): Height of the image in pixels.

    Returns:
        dict: A dictionary containing ground width, ground height, pixel width, and pixel height.
    """
    # Convert FoV from degrees to radians
    fov_horizontal_rad = math.radians(fov_horizontal)
    fov_vertical_rad = math.radians(fov_vertical)

    # Calculate ground coverage
    ground_width = 2 * altitude * math.tan(fov_horizontal_rad / 2)  # in meters
    ground_height = 2 * altitude * math.tan(fov_vertical_rad / 2)  # in meters

    # Calculate pixel resolution
    pixel_width = ground_width / image_width  # meters per pixel
    pixel_height = ground_height / image_height  # meters per pixel

    return {
        "ground_width": ground_width,
        "ground_height": ground_height,
        "pixel_width": pixel_width,
        "pixel_height": pixel_height,
    }

# Example usage
altitude = 30  # in meters
fov_horizontal = 82  # in degrees (example camera horizontal FoV)
fov_vertical = 54  # in degrees (example camera vertical FoV)
image_width = 4000  # in pixels
image_height = 3000  # in pixels

result = calculate_ground_coverage_and_resolution(
    altitude, fov_horizontal, fov_vertical, image_width, image_height
)

print("Ground Coverage and Pixel Resolution:")
print(f"Ground Width: {result['ground_width']} meters")
print(f"Ground Height: {result['ground_height']} meters")
print(f"Pixel Width: {result['pixel_width']} meters per pixel")
print(f"Pixel Height: {result['pixel_height']} meters per pixel")


Ground Coverage and Pixel Resolution:
Ground Width: 145.30850560107217 meters
Ground Height: 101.90508989888576 meters
Pixel Width: 0.03632712640026804 meters per pixel
Pixel Height: 0.03396836329962859 meters per pixel
